In [81]:
# ── Cell 1: Imports and paths ─────────────────────────────────────
import pandas as pd
import numpy as np
import pickle
import warnings
from pathlib import Path
from collections import defaultdict
from scipy.stats import poisson

warnings.filterwarnings('ignore')
np.random.seed(42)

BASE_DIR  = Path('')
RAW_DIR   = BASE_DIR / 'data' / 'raw'
PROC_DIR  = BASE_DIR / 'data' / 'processed'
MODEL_DIR = BASE_DIR / 'models'

# ── Load saved pipeline ───────────────────────────────────────────
with open(MODEL_DIR / 'models.pkl', 'rb') as f:
    saved = pickle.load(f)

final_ensemble = saved['final_ensemble']
FEATURE_COLS   = saved['feature_cols']
DRAW_THRESHOLD = saved['draw_threshold']

with open(MODEL_DIR / 'poisson_model.pkl', 'rb') as f:
    dc_model = pickle.load(f)

print(f'✅ Models loaded')
print(f'   Features      : {len(FEATURE_COLS)}')
print(f'   Draw threshold: {DRAW_THRESHOLD:.2f}')
print(f'   DC rho        : {dc_model["rho"]:.4f}')

✅ Models loaded
   Features      : 41
   Draw threshold: 0.29
   DC rho        : -0.1576


In [82]:
# ── Cell 2: Load data ─────────────────────────────────────────────
# Historical results (your existing dataset)
df_historical = pd.read_csv(RAW_DIR / 'results.csv', parse_dates=['date'])
df_historical = df_historical.dropna(
    subset=['home_score', 'away_score']
).sort_values('date').reset_index(drop=True)

# WC 2026 fixtures (includes R32 placeholders)
df_fixtures = pd.read_csv(
    RAW_DIR / 'wc_2026_fixtures.csv', parse_dates=['date']
)

# Teams and ELO snapshot
df_teams = pd.read_csv(RAW_DIR / 'wc_2026_teams.csv')
df_elo    = pd.read_csv(PROC_DIR / 'elo_all_teams.csv')
df_h2h    = pd.read_csv(PROC_DIR / 'h2h_records.csv')

# ELO lookup from pre-tournament snapshot
elo_lookup = dict(zip(df_elo['team'], df_elo['elo']))
team_fifa  = dict(zip(df_teams['team'], df_teams['fifa_rank']))
team_group = dict(zip(df_teams['team'], df_teams['group']))

WC26_TEAMS = df_teams['team'].tolist()

print(f'Historical matches : {len(df_historical):,}')
print(f'Fixtures           : {len(df_fixtures)}')
print(f'ELO snapshot teams : {len(elo_lookup)}')

Historical matches : 49,477
Fixtures           : 104
ELO snapshot teams : 326


In [83]:
# ── Cell 3: Pull WC 2026 completed results from martj42 ──────────
# The martj42 repo updates results.csv nightly
# We filter for WC 2026 matches that happened after tournament start

TOURNAMENT_START = pd.Timestamp('2026-06-11')

wc_2026_played = df_historical[
    (df_historical['date'] >= TOURNAMENT_START) &
    (df_historical['tournament'].str.contains('FIFA World Cup', na=False))
].copy().reset_index(drop=True)

print(f'WC 2026 completed matches : {len(wc_2026_played)}')
print(f'Date range: {wc_2026_played["date"].min()} → '
      f'{wc_2026_played["date"].max()}')
print()
print('Results so far:')
display(wc_2026_played[[
    'date', 'home_team', 'away_team',
    'home_score', 'away_score', 'tournament'
]])

WC 2026 completed matches : 72
Date range: 2026-06-11 00:00:00 → 2026-06-27 00:00:00

Results so far:


,date,home_team,away_team,home_score,away_score,tournament
0,2026-06-11,South Korea,Czech Republic,2.0,1.0,FIFA World Cup
1,2026-06-11,Mexico,South Africa,2.0,0.0,FIFA World Cup
2,2026-06-12,Canada,Bosnia and Herzegovina,1.0,1.0,FIFA World Cup
3,2026-06-12,United States,Paraguay,4.0,1.0,FIFA World Cup
4,2026-06-13,Qatar,Switzerland,1.0,1.0,FIFA World Cup
...,...,...,...,...,...,...
67,2026-06-27,Colombia,Portugal,0.0,0.0,FIFA World Cup
68,2026-06-27,Panama,England,0.0,2.0,FIFA World Cup
69,2026-06-27,Algeria,Austria,3.0,3.0,FIFA World Cup
70,2026-06-27,Jordan,Argentina,1.0,3.0,FIFA World Cup


In [84]:
# ── Cell 4: Update ELO with WC 2026 group stage results ──────────
from feature_engineering import get_k_factor  # or redefine below

def get_k_factor_wc(tournament, date):
    t = str(tournament).lower()
    if 'world cup' in t and 'qualif' not in t:
        return 60
    return 20

def update_elo_with_results(current_elos: dict,
                             df_new: pd.DataFrame,
                             home_advantage: int = 65) -> dict:
    """
    Walk through new completed matches and update ELO.
    All WC matches are neutral venue.
    """
    elos = current_elos.copy()

    for _, row in df_new.sort_values('date').iterrows():
        h, a   = row['home_team'], row['away_team']
        eh, ea = elos.get(h, 1500), elos.get(a, 1500)

        # WC = neutral venue
        exp_h = 1 / (1 + 10 ** ((ea - eh) / 400))
        exp_a = 1 - exp_h

        hs, as_ = row['home_score'], row['away_score']
        if   hs > as_: act_h, act_a = 1.0, 0.0
        elif hs < as_: act_h, act_a = 0.0, 1.0
        else:          act_h, act_a = 0.5, 0.5

        K = get_k_factor_wc(row['tournament'], row['date'])

        elos[h] = eh + K * (act_h - exp_h)
        elos[a] = ea + K * (act_a - exp_a)

    return elos

# Update ELO
updated_elos = update_elo_with_results(elo_lookup, wc_2026_played)

# Show ELO changes for WC teams
elo_changes = []
for team in WC26_TEAMS:
    before = elo_lookup.get(team, 1500)
    after  = updated_elos.get(team, 1500)
    elo_changes.append({
        'team':   team,
        'before': round(before, 1),
        'after':  round(after, 1),
        'change': round(after - before, 1),
    })

df_elo_changes = pd.DataFrame(elo_changes).sort_values(
    'change', ascending=False
).reset_index(drop=True)

print('ELO changes after group stage:')
display(df_elo_changes.head(48))

ELO changes after group stage:


,team,before,after,change
0,Ghana,1559.7,1615.9,56.2
1,Mexico,1892.6,1939.7,47.0
2,France,1973.4,2019.6,46.2
3,Cape Verde,1594.0,1639.9,45.9
4,South Africa,1628.1,1667.7,39.6
5,DR Congo,1681.6,1719.4,37.8
6,Norway,1799.5,1834.9,35.4
7,Morocco,1883.6,1917.0,33.4
8,Switzerland,1786.9,1819.4,32.5
9,Bosnia and Herzegovina,1585.7,1617.5,31.8


In [85]:
# ── Cell 5: Update rolling form with WC group stage matches ───────
def get_team_form_updated(team: str,
                           df_all_matches: pd.DataFrame,
                           windows: list = [3, 5, 10]) -> dict:
    """
    Compute form using all historical + WC 2026 group stage matches.
    """
    mask   = (
        (df_all_matches['home_team'] == team) |
        (df_all_matches['away_team'] == team)
    )
    recent = (
        df_all_matches[mask]
        .dropna(subset=['home_score', 'away_score'])
        .sort_values('date')
        .reset_index(drop=True)
    )

    if len(recent) == 0:
        return {f'{s}_{w}': v for w in windows
                for s, v in [('win_rate', 0.5), ('avg_gf', 1.0),
                              ('avg_ga', 1.0), ('avg_gd', 0.0)]}

    is_home = (recent['home_team'] == team).values
    gf   = np.where(is_home, recent['home_score'],
                    recent['away_score']).astype(float)
    ga   = np.where(is_home, recent['away_score'],
                    recent['home_score']).astype(float)
    wins = (gf > ga).astype(float)

    out = {}
    for w in windows:
        lgf, lga, lw = gf[-w:], ga[-w:], wins[-w:]
        n = len(lgf)
        out[f'win_rate_{w}'] = lw.mean()  if n > 0 else 0.5
        out[f'avg_gf_{w}']   = lgf.mean() if n > 0 else 1.0
        out[f'avg_ga_{w}']   = lga.mean() if n > 0 else 1.0
        out[f'avg_gd_{w}']   = out[f'avg_gf_{w}'] - out[f'avg_ga_{w}']
    return out

# Combine historical + WC group stage for form calculation
df_all_with_wc = pd.concat([
    df_historical, wc_2026_played
]).drop_duplicates(
    subset=['date', 'home_team', 'away_team']
).sort_values('date').reset_index(drop=True)

print('Computing updated form for all WC 2026 teams...')
team_form_updated = {
    t: get_team_form_updated(t, df_all_with_wc)
    for t in WC26_TEAMS
}
print('✅ Form updated')

# Spot check
print('\nFrance updated form:')
print(team_form_updated.get('France', {}))

Computing updated form for all WC 2026 teams...
✅ Form updated

France updated form:
{'win_rate_3': np.float64(1.0), 'avg_gf_3': np.float64(3.3333333333333335), 'avg_ga_3': np.float64(0.6666666666666666), 'avg_gd_3': np.float64(2.666666666666667), 'win_rate_5': np.float64(0.8), 'avg_gf_5': np.float64(2.8), 'avg_ga_5': np.float64(1.0), 'avg_gd_5': np.float64(1.7999999999999998), 'win_rate_10': np.float64(0.8), 'avg_gf_10': np.float64(2.8), 'avg_ga_10': np.float64(1.0), 'avg_gd_10': np.float64(1.7999999999999998)}


In [86]:
# ── Cell 7: Top scorers from martj42 data ─────────────────────────
# martj42 repo has a goalscorers.csv with player-level goals
# Try to load it, fall back to aggregating from results if absent

scorers_path = RAW_DIR / 'goalscorers.csv'

if scorers_path.exists():
    df_scorers_raw = pd.read_csv(scorers_path, parse_dates=['date'])

    # Filter WC 2026 group stage
    wc_scorers = df_scorers_raw[
        df_scorers_raw['date'] >= TOURNAMENT_START
    ].copy()

    # Aggregate goals per player
    top_scorers = (
        wc_scorers[wc_scorers['own_goal'] == False]  # exclude OGs
        .groupby(['scorer', 'team'])
        .agg(
            goals   = ('scorer',   'count'),
            penalty = ('penalty',  'sum'),
        )
        .reset_index()
        .sort_values('goals', ascending=False)
        .reset_index(drop=True)
    )
    top_scorers.index += 1

    print('🏆 WC 2026 Top Scorers:')
    display(top_scorers.head(20))
    top_scorers.to_csv(PROC_DIR / 'wc2026_top_scorers.csv', index=False)
    print('✅ Saved wc2026_top_scorers.csv')

    # Own goals
    og = wc_scorers[wc_scorers['own_goal'] == True]
    if len(og) > 0:
        print(f'\nOwn goals: {len(og)}')
        display(og[['date', 'home_team', 'away_team', 'team', 'scorer']])
else:
    print('⚠️  goalscorers.csv not found in data/raw/')
    print('   Download from: https://github.com/martj42/international_results')
    print('   File: goalscorers.csv')
    print()
    # Fallback — we know goals from results but not scorers
    print('Goal tallies from results only (no player names):')
    home_goals = wc_2026_played.groupby('home_team')['home_score'].sum()
    away_goals = wc_2026_played.groupby('away_team')['away_score'].sum()
    total_goals = (home_goals.add(away_goals, fill_value=0)
                   .sort_values(ascending=False)
                   .reset_index())
    total_goals.columns = ['team', 'goals']
    display(total_goals)

🏆 WC 2026 Top Scorers:


,scorer,team,goals,penalty
1,Lionel Messi,Argentina,6,0
2,Erling Haaland,Norway,4,0
3,Ousmane Dembélé,France,4,0
4,Kylian Mbappé,France,4,0
5,Vinícius Júnior,Brazil,4,0
6,Ismaïla Sarr,Senegal,3,0
7,Deniz Undav,Germany,3,0
8,Brian Brobbey,Netherlands,3,0
9,Elijah Just,New Zealand,3,0
10,Jonathan David,Canada,3,0


✅ Saved wc2026_top_scorers.csv

Own goals: 12


,date,home_team,away_team,team,scorer
47613,2026-06-12,United States,Paraguay,United States,Damián Bobadilla
47624,2026-06-13,Qatar,Switzerland,Qatar,Miro Muheim
47645,2026-06-15,Belgium,Egypt,Belgium,Mohamed Hany
47657,2026-06-16,Austria,Jordan,Austria,Yazan Al-Arab
47667,2026-06-16,Iraq,Norway,Norway,Aymen Hussein
47685,2026-06-18,Canada,Qatar,Canada,Mohamed Manai
47700,2026-06-19,United States,Australia,United States,Cameron Burgess
47722,2026-06-21,Spain,Saudi Arabia,Spain,Hassan Al-Tambakti
47745,2026-06-23,Portugal,Uzbekistan,Portugal,Abduvohid Nematov
47748,2026-06-24,Bosnia and Herzegovina,Qatar,Bosnia and Herzegovina,Mahmud Abunada


In [87]:
# ── Cell 8: Build R32 fixture list with actual teams ──────────────
# Get group standings from completed results
def compute_group_standings(df_results: pd.DataFrame,
                             team_group: dict) -> dict:
    """
    Compute group standings from WC 2026 results.
    Returns dict {group: [1st, 2nd, 3rd, 4th]} sorted by pts/gd/gf.
    """
    rows = []
    for _, m in df_results.iterrows():
        for side, opp_side, gf_col, ga_col in [
            ('home_team', 'away_team', 'home_score', 'away_score'),
            ('away_team', 'home_team', 'away_score', 'home_score'),
        ]:
            team = m[side]
            gf   = m[gf_col]
            ga   = m[ga_col]
            rows.append({
                'team': team,
                'group': team_group.get(team, '?'),
                'gf':  int(gf),
                'ga':  int(ga),
                'gd':  int(gf - ga),
                'pts': 3 if gf > ga else (1 if gf == ga else 0),
                'w':   1 if gf > ga else 0,
                'd':   1 if gf == ga else 0,
                'l':   1 if gf < ga else 0,
                'mp':  1,
            })

    standings = (
        pd.DataFrame(rows)
        .groupby(['team', 'group'])
        .sum()
        .reset_index()
        .sort_values(
            ['group', 'pts', 'gd', 'gf'],
            ascending=[True, False, False, False]
        )
    )

    group_dict = {}
    for g in sorted(standings['group'].unique()):
        if g == '?':
            continue
        grp = standings[standings['group'] == g]
        group_dict[g] = grp['team'].tolist()

    return group_dict, standings

team_group['United States'] = 'D' 


group_standings, df_standings = compute_group_standings(
    wc_2026_played, team_group
)

print('Group Standings:')
display(df_standings[[
    'group', 'team', 'mp', 'w', 'd', 'l',
    'gf', 'ga', 'gd', 'pts'
]].sort_values(['group', 'pts', 'gd'], ascending=[True, False, False]))

# Best 8 third-place teams
thirds = []
for g, teams_ranked in group_standings.items():
    if len(teams_ranked) >= 3:
        third_team = teams_ranked[2]
        row = df_standings[df_standings['team'] == third_team].iloc[0]
        thirds.append({
            'team':  third_team,
            'group': g,
            'pts':   int(row['pts']),
            'gd':    int(row['gd']),
            'gf':    int(row['gf']),
        })

thirds_df = pd.DataFrame(thirds).sort_values(
    ['pts', 'gd', 'gf'], ascending=False
).reset_index(drop=True)

thirds_df.index = thirds_df.index + 1

best_thirds = thirds_df.head(8)['team'].tolist()

print(f'\nBest 8 third-place teams:')
display(thirds_df)
print(f'Advancing: {best_thirds}')

Group Standings:


,group,team,mp,w,d,l,gf,ga,gd,pts
26,A,Mexico,3,3,0,0,6,0,6,9
38,A,South Africa,3,1,1,1,2,3,-1,4
39,A,South Korea,3,1,0,2,2,3,-1,3
12,A,Czech Republic,3,0,1,2,2,6,-4,1
42,B,Switzerland,3,2,1,0,7,3,4,7
7,B,Canada,3,1,1,1,8,3,5,4
5,B,Bosnia and Herzegovina,3,1,1,1,5,6,-1,4
34,B,Qatar,3,0,1,2,2,10,-8,1
6,C,Brazil,3,2,1,0,7,1,6,7
27,C,Morocco,3,2,1,0,6,3,3,7



Best 8 third-place teams:


,team,group,pts,gd,gf
1,DR Congo,K,4,1,4
2,Sweden,F,4,0,7
3,Ecuador,E,4,0,2
4,Ghana,L,4,0,2
5,Bosnia and Herzegovina,B,4,-1,5
6,Algeria,J,4,-2,5
7,Paraguay,D,4,-2,2
8,Senegal,I,3,2,8
9,Iran,G,3,0,3
10,South Korea,A,3,-1,2


Advancing: ['DR Congo', 'Sweden', 'Ecuador', 'Ghana', 'Bosnia and Herzegovina', 'Algeria', 'Paraguay', 'Senegal']


In [88]:
r32_raw= pd.read_csv(f'{RAW_DIR}/results.csv').tail(16)
display(r32_raw)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49477,2026-06-28,South Africa,Canada,NaN,NaN,FIFA World Cup,Inglewood,United States,True
49478,2026-06-29,Brazil,Japan,NaN,NaN,FIFA World Cup,Houston,United States,True
49479,2026-06-29,Germany,Paraguay,NaN,NaN,FIFA World Cup,Foxborough,United States,True
49480,2026-06-29,Netherlands,Morocco,NaN,NaN,FIFA World Cup,Guadalupe,Mexico,True
49481,2026-06-30,Ivory Coast,Norway,NaN,NaN,FIFA World Cup,Arlington,United States,True
49482,2026-06-30,France,Sweden,NaN,NaN,FIFA World Cup,East Rutherford,United States,True
49483,2026-06-30,Mexico,Ecuador,NaN,NaN,FIFA World Cup,Mexico City,Mexico,False
49484,2026-07-01,England,DR Congo,NaN,NaN,FIFA World Cup,Atlanta,United States,True
49485,2026-07-01,Belgium,Senegal,NaN,NaN,FIFA World Cup,Seattle,United States,True
49486,2026-07-01,United States,Bosnia and Herzegovina,NaN,NaN,FIFA World Cup,Santa Clara,United States,False


In [89]:
# ── Cell 9: Build R32 matchups from actual schedule ──────────────
# The R32 consists of 16 matches. We grab the last 16 rows from 
# the chronological results dataset.

# Note: Replace 'wc_2026_played' with whatever your full results dataframe is named 
# if you haven't split it yet.


# Filter and rename the columns to match your desired structure
df_r32 = r32_raw[['home_team', 'away_team']].rename(
    columns={'home_team': 'home', 'away_team': 'away'}
)

# Add a clean label and reset the index
df_r32['label'] = [f'Round of 32 - Match {i+1}' for i in range(16)]
df_r32 = df_r32.reset_index(drop=True)

# Shift index to 1-based for display purposes (optional)
df_r32.index = df_r32.index + 1

print(f'Round of 32 matchups ({len(df_r32)} matches):')
display(df_r32)

Round of 32 matchups (16 matches):


,home,away,label
1,South Africa,Canada,Round of 32 - Match 1
2,Brazil,Japan,Round of 32 - Match 2
3,Germany,Paraguay,Round of 32 - Match 3
4,Netherlands,Morocco,Round of 32 - Match 4
5,Ivory Coast,Norway,Round of 32 - Match 5
6,France,Sweden,Round of 32 - Match 6
7,Mexico,Ecuador,Round of 32 - Match 7
8,England,DR Congo,Round of 32 - Match 8
9,Belgium,Senegal,Round of 32 - Match 9
10,United States,Bosnia and Herzegovina,Round of 32 - Match 10


In [90]:
TEAM_NAME_MAP = {
    'United States': 'USA',
}

df_r32['home'] = df_r32['home'].replace(TEAM_NAME_MAP)
df_r32['away'] = df_r32['away'].replace(TEAM_NAME_MAP)

# Verify
r32_teams     = set(df_r32['home'].tolist() + df_r32['away'].tolist())
still_missing = r32_teams - set(team_form_updated.keys())
if still_missing:
    print(f"❌ Still missing: {still_missing}")
else:
    print(f"✅ All R32 teams matched — safe to run prediction loop")
print("USA in updated_elos:", 'USA' in updated_elos or 'United States' in updated_elos)
print("USA in team_fifa:   ", 'USA' in team_fifa    or 'United States' in team_fifa)

✅ All R32 teams matched — safe to run prediction loop
USA in updated_elos: True
USA in team_fifa:    True


In [91]:
def build_pen_rates(df_shootouts: pd.DataFrame,
                    min_games: int = 2) -> dict:
    """
    Compute per-team penalty shootout win rate from historical data.
    Returns dict: {team_name: win_rate}  (default 0.5 for unseen teams)
    """
    records = []
    for _, row in df_shootouts.iterrows():
        home   = row['home_team']
        away   = row['away_team']
        winner = row['winner']
        records.append({'team': home, 'won': int(home == winner)})
        records.append({'team': away, 'won': int(away == winner)})

    df_pen = pd.DataFrame(records)
    pen_stats = (
        df_pen.groupby('team')['won']
        .agg(wins='sum', games='count')
        .assign(pen_win_rate=lambda x: x['wins'] / x['games'])
        .sort_values('games', ascending=False)
    )

    # Only trust teams with enough history; rest default to 0.5 at lookup
    reliable = pen_stats[pen_stats['games'] >= min_games]
    print(f"Shootout records : {len(df_shootouts)} matches")
    print(f"Teams with data  : {len(pen_stats)} total, "
          f"{len(reliable)} with >= {min_games} shootouts")
    print("\nTop 20 by appearances:")
    print(reliable.head(20).to_string())

    return pen_stats['pen_win_rate'].to_dict()


df_shootouts = pd.read_csv('data/raw/shootouts.csv')
pen_rates    = build_pen_rates(df_shootouts, min_games=2)


def resolve_knockout_winner(p_h: float, p_d: float, p_a: float,
                             home: str, away: str,
                             pen_rates: dict) -> tuple:
    """
    Splits draw probability using team-specific historical pen win rates.
    Normalises so home + away pen shares always sum to 1.
    Falls back to 0.5/0.5 for teams with no shootout history.
    """
    h_pen  = pen_rates.get(home, 0.5)
    a_pen  = pen_rates.get(away, 0.5)
    total  = h_pen + a_pen
    h_norm = h_pen / total
    a_norm = a_pen / total

    p_h_ko = p_h + p_d * h_norm
    p_a_ko = p_a + p_d * a_norm
    winner = home if p_h_ko > p_a_ko else away

    return winner, round(p_h_ko * 100, 1), round(p_a_ko * 100, 1), round(h_norm * 100, 1)


# ── Spot check pen rates for WC 2026 R32 teams ────────────────────
r32_teams = set(df_r32['home'].tolist() + df_r32['away'].tolist())
print("\nPen rates for R32 teams:")
for team in sorted(r32_teams):
    rate  = pen_rates.get(team, None)
    flag  = '(default 0.5)' if rate is None else ''
    print(f"  {team:<25} {rate if rate else 0.5:.3f}  {flag}")

Shootout records : 678 matches
Teams with data  : 230 total, 187 with >= 2 shootouts

Top 20 by appearances:
              wins  games  pen_win_rate
team                                   
South Africa    13     28      0.464286
Egypt           15     26      0.576923
South Korea     15     25      0.600000
Zambia          14     25      0.560000
Iran            10     23      0.434783
Argentina       15     23      0.652174
Thailand        13     22      0.590909
Senegal         11     21      0.523810
Mali             8     18      0.444444
Uganda           7     18      0.388889
Malawi           4     18      0.222222
Ivory Coast     10     18      0.555556
Cameroon         9     18      0.500000
Nigeria         10     17      0.588235
Uruguay          9     17      0.529412
Kenya           10     16      0.625000
Brazil           9     16      0.562500
Gabon            5     15      0.333333
Zimbabwe         7     15      0.466667
Algeria          7     15      0.466667

Pen rates 

In [92]:
# ── Restart check ─────────────────────────────────────────────────────
# Confirm no stale definition exists before running
import inspect

DEFAULT_FORM = {
    'win_rate_3':  0.4,  'avg_gf_3':  1.0,  'avg_ga_3':  1.0,  'avg_gd_3':  0.0,
    'win_rate_5':  0.4,  'avg_gf_5':  1.0,  'avg_ga_5':  1.0,  'avg_gd_5':  0.0,
    'win_rate_10': 0.4,  'avg_gf_10': 1.0,  'avg_ga_10': 1.0,  'avg_gd_10': 0.0,
}

def build_match_features(home: str, away: str,
                          updated_elos: dict,
                          team_form: dict,
                          df_h2h: pd.DataFrame,
                          team_fifa: dict,
                          feature_cols: list) -> pd.DataFrame:

    h_elo = updated_elos.get(home, 1500)
    a_elo = updated_elos.get(away, 1500)

    # Always has all keys — real values override where available
    hf = {**DEFAULT_FORM, **team_form.get(home, {})}
    af = {**DEFAULT_FORM, **team_form.get(away, {})}

    # Team name mismatch diagnostic
    missing = [t for t in [home, away] if t not in team_form]
    if missing:
        print(f"  [WARN] No form data for {missing} — check team name spelling vs WC26_TEAMS")

    # H2H
    r = df_h2h[(df_h2h['team_a'] == home) & (df_h2h['team_b'] == away)]
    if len(r):
        rh = r.iloc[0]
        h2h = {
            'h2h_games':           int(rh['games']),
            'h2h_home_wins':       int(rh['a_wins']),
            'h2h_draws':           int(rh['draws']),
            'h2h_away_wins':       int(rh['b_wins']),
            'h2h_home_goals':      float(rh['a_goals']),
            'h2h_away_goals':      float(rh['b_goals']),
            'h2h_avg_goal_diff':   float(rh['avg_goal_diff']),
            'h2h_home_win_rate':   float(rh['a_win_rate']),
            'h2h_home_win_rate_w': float(rh['a_win_rate_weighted'])
                                   if pd.notna(rh['a_win_rate_weighted']) else 0.5,
        }
    else:
        h2h = {
            'h2h_games': 0,           'h2h_home_wins': 0,       'h2h_draws': 0,
            'h2h_away_wins': 0,       'h2h_home_goals': 0.0,    'h2h_away_goals': 0.0,
            'h2h_avg_goal_diff': 0.0, 'h2h_home_win_rate': 0.5, 'h2h_home_win_rate_w': 0.5,
        }

    h_rank = team_fifa.get(home, 100)
    a_rank = team_fifa.get(away, 100)

    row = {
        'home_elo_before': h_elo,
        'away_elo_before': a_elo,
        'elo_diff':        h_elo - a_elo,
        'home_elo_rank':   h_rank,
        'away_elo_rank':   a_rank,
        'rank_diff':       h_rank - a_rank,
        **{f'home_{k}': v for k, v in hf.items()},
        **{f'away_{k}': v for k, v in af.items()},
        **{f'win_rate_diff_{w}':
           hf.get(f'win_rate_{w}', 0.4) - af.get(f'win_rate_{w}', 0.4)
           for w in [3, 5, 10]},
        **{f'gd_diff_{w}':
           hf.get(f'avg_gd_{w}', 0.0) - af.get(f'avg_gd_{w}', 0.0)
           for w in [3, 5, 10]},
        **h2h,
        'is_neutral': 1,
    }

    return pd.DataFrame([row]).reindex(columns=feature_cols).fillna(0)


# ── Verify the correct function is now active ─────────────────────────
src = inspect.getsource(build_match_features)
assert 'reindex' in src,      "❌ Old function still active — reindex missing"
assert 'DEFAULT_FORM' in src, "❌ Old function still active — DEFAULT_FORM missing"
print("✅ build_match_features correctly defined")

# ── Verify all R32 teams are in team_form_updated ─────────────────────
r32_teams = set(df_r32['home'].tolist() + df_r32['away'].tolist())
missing_form = r32_teams - set(team_form_updated.keys())
if missing_form:
    print(f"❌ Teams in df_r32 missing from team_form_updated: {missing_form}")
    print(f"   WC26_TEAMS sample: {sorted(list(team_form_updated.keys()))[:5]}")
else:
    print(f"✅ All {len(r32_teams)} R32 teams have form data")

✅ build_match_features correctly defined
✅ All 32 R32 teams have form data


In [94]:

def predict_r32_match(home: str, away: str,
                       updated_elos: dict,
                       team_form: dict,
                       df_h2h: pd.DataFrame,
                       team_fifa: dict,
                       feature_cols: list,
                       ensemble,
                       dc_model: dict,
                       dc_weight: float = 0.6) -> dict:
    """
    Full prediction for a single R32 match.
    Blends ensemble + Dixon-Coles.
    """
    X = build_match_features(
        home, away, updated_elos, team_form,
        df_h2h, team_fifa, feature_cols
    )

    # Ensemble probabilities
    probs = ensemble.predict_proba(X)[0]
    probs = probs / probs.sum()
    en_home = float(probs[2])
    en_draw = float(probs[1])
    en_away = float(probs[0])

    # Dixon-Coles scoreline
    att   = dc_model['attack']
    defe  = dc_model['defense']
    rho   = dc_model['rho']
    d_att = dc_model.get('default_attack', 0.0)
    d_def = dc_model.get('default_defense', 0.0)
    mg    = int(dc_model.get('max_goals', 8))

    # Knockout rho adjustment
    rho_ko = max(min(rho * 1.15, 0.0), -0.18)

    lh = np.exp(att.get(home, d_att) - defe.get(away, d_def))
    la = np.exp(att.get(away, d_att) - defe.get(home, d_def))

    gi = np.arange(mg)
    m  = np.outer(poisson.pmf(gi, lh), poisson.pmf(gi, la))
    m[0,0] *= 1 - lh * la * rho_ko
    m[1,0] *= 1 + la * rho_ko
    m[0,1] *= 1 + lh * rho_ko
    m[1,1] *= 1 - rho_ko
    np.clip(m, 0.0, None, out=m)
    m /= m.sum()

    dc_home = float(np.tril(m, -1).sum())
    dc_draw = float(np.diag(m).sum())
    dc_away = float(np.triu(m, 1).sum())

    # Blend
    en_w = 1 - dc_weight
    p_h  = dc_weight * dc_home + en_w * en_home
    p_d  = dc_weight * dc_draw + en_w * en_draw
    p_a  = dc_weight * dc_away + en_w * en_away
    s    = p_h + p_d + p_a
    p_h, p_d, p_a = p_h/s, p_d/s, p_a/s

    # Rescale matrix
    n = m.shape[0]
    for mask, target in (
        (np.tril(np.ones((n,n)),-1).astype(bool), p_h),
        (np.eye(n, dtype=bool), p_d),
        (np.triu(np.ones((n,n)),1).astype(bool), p_a),
    ):
        cur = m[mask].sum()
        if cur > 0:
            m[mask] *= target / cur
    m /= m.sum()

    # Top scorelines
    flat = sorted(
        ((i, j, m[i,j]) for i in range(mg) for j in range(mg)),
        key=lambda x: -x[2]
    )
    top5 = [(f'{i}-{j}', round(float(p)*100, 1)) for i,j,p in flat[:5]]

    grid = np.add.outer(np.arange(mg), np.arange(mg))

    winner, p_h_ko, p_a_ko, h_pen_pct = resolve_knockout_winner(
        p_h, p_d, p_a, home, away, pen_rates
    )

    return {
        'home':             home,
        'away':             away,
        'p_home_win_90':    round(p_h * 100, 1),   # regulation
        'p_draw_90':        round(p_d * 100, 1),   # → pens
        'p_away_win_90':    round(p_a * 100, 1),   # regulation
        'p_home_win_ko':    p_h_ko,                 # inc. pen share
        'p_away_win_ko':    p_a_ko,                 # inc. pen share
        'pen_adv_home_pct': h_pen_pct,
        'predicted_winner': winner,                  # pen-aware
        'xg_home':          round(float(lh), 2),
        'xg_away':          round(float(la), 2),
        'predicted_score':  f'{round(lh)}-{round(la)}',
        'top_scorelines':   top5,
        'p_over25':         round(float(m[grid > 2].sum()) * 100, 1),
        'p_btts':           round(float(1 - m[0,:].sum() - m[:,0].sum() + m[0,0]) * 100, 1),
        'p_home_cs':        round(float(m[:,0].sum()) * 100, 1),
        'p_away_cs':        round(float(m[0,:].sum()) * 100, 1),
    }


# ── Run predictions for all R32 matches ──────────────────────────
print('⚙️  Predicting Round of 32 matches...\n')
r32_predictions = []

for _, fix in df_r32.iterrows():
    home = fix['home']
    away = fix['away']

    pred = predict_r32_match(
        home, away,
        updated_elos, team_form_updated,
        df_h2h, team_fifa,
        FEATURE_COLS, final_ensemble, dc_model
    )
    pred['label'] = fix['label']
    r32_predictions.append(pred)

df_r32_preds = pd.DataFrame(r32_predictions)

# Display
print('='*70)
print('ROUND OF 32 PREDICTIONS')
print('='*70)
# NEW
for _, row in df_r32_preds.iterrows():
    winner_flag_h = '🏆' if row['predicted_winner'] == row['home'] else '  '
    winner_flag_a = '🏆' if row['predicted_winner'] == row['away'] else '  '
    print(f"\n  {winner_flag_h}{row['home']:<22} vs "
          f"{row['away']:<22}{winner_flag_a}")
    print(f"     90 min  — {row['home']}: {row['p_home_win_90']}%  |  "
          f"Draw: {row['p_draw_90']}%  |  "
          f"{row['away']}: {row['p_away_win_90']}%")
    print(f"     KO prob — {row['home']}: {row['p_home_win_ko']}%  |  "
          f"{row['away']}: {row['p_away_win_ko']}%  "
          f"(pen adv home: {row['pen_adv_home_pct']}%)")
    print(f"     Predicted score : {row['predicted_score']}  "
          f"(xG: {row['xg_home']} – {row['xg_away']})")
    print(f"     Top scorelines  : "
          f"{' | '.join([f'{s} ({p}%)' for s,p in row['top_scorelines'][:3]])}")
    print(f"     Over 2.5: {row['p_over25']}%  "
          f"BTTS: {row['p_btts']}%")

# Save
display(df_r32_preds[[
    'home', 'away', 'predicted_winner',
    'p_home_win_90', 'p_draw_90', 'p_away_win_90',
    'p_home_win_ko', 'p_away_win_ko', 'pen_adv_home_pct',
    'predicted_score', 'xg_home', 'xg_away',
    'p_over25', 'p_btts'
]])
df_r32_preds.to_csv(PROC_DIR / 'r32_predictions.csv', index=False)
print('\n✅ Saved r32_predictions.csv')

⚙️  Predicting Round of 32 matches...

ROUND OF 32 PREDICTIONS

    South Africa           vs Canada                🏆
     90 min  — South Africa: 34.8%  |  Draw: 23.6%  |  Canada: 41.6%
     KO prob — South Africa: 47.9%  |  Canada: 52.1%  (pen adv home: 55.3%)
     Predicted score : 1-2  (xG: 0.91 – 1.67)
     Top scorelines  : 1-1 (11.3%) | 2-1 (10.0%) | 1-0 (9.2%)
     Over 2.5: 49.0%  BTTS: 50.5%

    Brazil                 vs Japan                 🏆
     90 min  — Brazil: 34.7%  |  Draw: 26.0%  |  Japan: 39.3%
     KO prob — Brazil: 49.6%  |  Japan: 50.4%  (pen adv home: 57.4%)
     Predicted score : 1-2  (xG: 0.79 – 1.55)
     Top scorelines  : 1-1 (12.2%) | 1-0 (11.2%) | 0-0 (10.3%)
     Over 2.5: 42.2%  BTTS: 45.6%

  🏆Germany                vs Paraguay                
     90 min  — Germany: 69.1%  |  Draw: 21.5%  |  Paraguay: 9.4%
     KO prob — Germany: 82.4%  |  Paraguay: 17.6%  (pen adv home: 61.9%)
     Predicted score : 3-1  (xG: 2.74 – 0.74)
     Top scorelines  : 2-0 

,home,away,predicted_winner,p_home_win_90,p_draw_90,p_away_win_90,p_home_win_ko,p_away_win_ko,pen_adv_home_pct,predicted_score,xg_home,xg_away,p_over25,p_btts
0,South Africa,Canada,Canada,34.8,23.6,41.6,47.9,52.1,55.3,1-2,0.91,1.67,49.0,50.5
1,Brazil,Japan,Japan,34.7,26.0,39.3,49.6,50.4,57.4,1-2,0.79,1.55,42.2,45.6
2,Germany,Paraguay,Germany,69.1,21.5,9.4,82.4,17.6,61.9,3-1,2.74,0.74,64.2,52.4
3,Netherlands,Morocco,Morocco,37.4,30.9,31.7,47.2,52.8,31.8,1-1,1.00,1.06,35.3,42.8
4,Ivory Coast,Norway,Norway,19.3,21.9,58.7,30.9,69.1,52.6,1-3,0.97,2.66,67.9,61.5
5,France,Sweden,France,73.0,17.0,10.0,79.9,20.1,40.5,3-1,2.64,1.14,73.6,63.6
6,Mexico,Ecuador,Mexico,37.0,48.5,14.4,61.3,38.7,50.0,0-0,0.44,0.23,3.7,7.5
7,England,DR Congo,England,74.9,18.8,6.3,81.5,18.5,34.8,2-0,2.17,0.24,41.1,21.0
8,Belgium,Senegal,Belgium,39.9,26.2,33.9,57.1,42.9,65.6,2-2,1.74,1.82,68.6,70.8
9,USA,Bosnia and Herzegovina,Bosnia and Herzegovina,11.8,24.6,63.6,21.7,78.3,40.0,0-2,0.14,1.56,22.5,12.0



✅ Saved r32_predictions.csv


In [96]:
# ── Cell 11: Compare R32 predictions vs actuals (when played) ─────
r32_completed = df_historical[
    (df_historical['date'] >= pd.Timestamp('2026-06-27')) &
    (df_historical['tournament'].str.contains('FIFA World Cup', na=False))
].copy()

if len(r32_completed) == 0:
    print('⏳ R32 matches not yet played — run this cell after they complete')
else:
    r32_completed['match_key'] = (
        r32_completed['home_team'] + '_' + r32_completed['away_team']
    )
    df_r32_preds['match_key'] = (
        df_r32_preds['home'] + '_' + df_r32_preds['away']
    )

    r32_comp = df_r32_preds.merge(
        r32_completed[['match_key', 'home_score', 'away_score']],
        on='match_key', how='inner'
    )

    # ── Actual result (90 min) ─────────────────────────────────────
    r32_comp['actual_result'] = np.where(
        r32_comp['home_score'] > r32_comp['away_score'], 'home',
        np.where(r32_comp['home_score'] < r32_comp['away_score'],
                 'away', 'draw')
    )

    # ── Predicted result based on 90-min probs ────────────────────
    r32_comp['predicted_result'] = np.where(
        r32_comp['p_home_win_90'] > r32_comp['p_away_win_90'],
        np.where(r32_comp['p_home_win_90'] > r32_comp['p_draw_90'],
                 'home', 'draw'),
        np.where(r32_comp['p_away_win_90'] > r32_comp['p_draw_90'],
                 'away', 'draw')
    )

    r32_comp['result_correct'] = (
        r32_comp['predicted_result'] == r32_comp['actual_result']
    ).astype(int)

    r32_comp['score_correct'] = (
        (r32_comp['predicted_score'].str.split('-').str[0].astype(int) ==
         r32_comp['home_score']) &
        (r32_comp['predicted_score'].str.split('-').str[1].astype(int) ==
         r32_comp['away_score'])
    ).astype(int)

    # ── Actual KO winner (accounts for shootout if draw) ──────────
    r32_comp['actual_winner_90'] = np.where(
        r32_comp['home_score'] > r32_comp['away_score'],
        r32_comp['home'], r32_comp['away']
    )  # for draws this will be wrong — update manually once shootout results known

    # ── Winner accuracy: KO prob-based predicted_winner vs actual ─
    r32_comp['winner_correct_ko'] = (
        r32_comp['predicted_winner'] == r32_comp['actual_winner_90']
    ).astype(int)

    print('='*60)
    print('ROUND OF 32 — PREDICTIONS VS ACTUALS')
    print('='*60)
    print(f'Matches evaluated      : {len(r32_comp)}')
    print(f'90-min result accuracy : {r32_comp["result_correct"].mean():.1%}')
    print(f'KO winner accuracy     : {r32_comp["winner_correct_ko"].mean():.1%}')
    print(f'Exact score hits       : {r32_comp["score_correct"].sum()}')

    display(r32_comp[[
        'home', 'away',
        'home_score', 'away_score',
        'predicted_score',
        'p_home_win_90', 'p_draw_90', 'p_away_win_90',
        'p_home_win_ko', 'p_away_win_ko', 'pen_adv_home_pct',
        'predicted_winner', 'actual_winner_90',
        'result_correct', 'winner_correct_ko', 'score_correct',
    ]])

    r32_comp.to_csv(PROC_DIR / 'r32_comparison.csv', index=False)
    print('✅ Saved r32_comparison.csv')

ROUND OF 32 — PREDICTIONS VS ACTUALS
Matches evaluated      : 0
90-min result accuracy : nan%
KO winner accuracy     : nan%
Exact score hits       : 0


,home,away,home_score,away_score,predicted_score,p_home_win_90,p_draw_90,p_away_win_90,p_home_win_ko,p_away_win_ko,pen_adv_home_pct,predicted_winner,actual_winner_90,result_correct,winner_correct_ko,score_correct


✅ Saved r32_comparison.csv


In [97]:
import json

def save_r32_predictions(df_r32_preds: pd.DataFrame, out_path: Path):
    """
    Serialise R32 predictions to JSON in dashboard-compatible format.
    top_scorelines stored as list of dicts {score, prob}.
    """
    records = []
    for _, row in df_r32_preds.iterrows():
        # Normalise top_scorelines to list of dicts
        raw_scores = row['top_scorelines']
        if isinstance(raw_scores, list) and len(raw_scores):
            # handles both tuple ('1-0', 9.2) and dict {'score','prob'} formats
            if isinstance(raw_scores[0], dict):
                top_scores = raw_scores
            else:
                top_scores = [{'score': s, 'prob': p} for s, p in raw_scores]
        else:
            top_scores = []

        records.append({
            # identifiers
            'home':              row['home'],
            'away':              row['away'],
            'label':             row.get('label', ''),
            # 90-min probs (used for scoreline context)
            'p_home_win':        row['p_home_win_90'],   # dashboard key
            'p_draw':            row['p_draw_90'],        # dashboard key
            'p_away_win':        row['p_away_win_90'],    # dashboard key
            # KO probs (inc. penalty share)
            'p_home_win_ko':     row['p_home_win_ko'],
            'p_away_win_ko':     row['p_away_win_ko'],
            'pen_adv_home_pct':  row['pen_adv_home_pct'],
            # winner
            'predicted_winner':  row['predicted_winner'],
            # scoreline
            'xg_home':           row['xg_home'],
            'xg_away':           row['xg_away'],
            'predicted_score':   row['predicted_score'],
            'top_scorelines':    top_scores,
            # market stats
            'p_over25':          row['p_over25'],
            'p_btts':            row['p_btts'],
            'p_home_cs':         row['p_home_cs'],
            'p_away_cs':         row['p_away_cs'],
        })

    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, 'w') as f:
        json.dump(records, f, indent=2)
    print(f'✅ Saved {len(records)} R32 predictions → {out_path}')


save_r32_predictions(df_r32_preds, PROC_DIR / 'r32_predictions.json')

✅ Saved 16 R32 predictions → data\processed\r32_predictions.json
